In [35]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath('.')))

from sqlalchemy import text
from db.database import engine

query = """
    SELECT 
        iso_code, 
        country, 
        verdict, 
        caveat 
    FROM 
        decoupler_class
"""

with engine.connect() as conn:
    classes = pd.read_sql(
        text(query),
        conn
    )

# Map decoupler_class verdicts (built in NB04) -> the three display groups this notebook compares.
# genuine -> Genuine
# fake -> Fake
# net_exporter -> Special  (e.g. Poland - the "misunderstood emitter")
# no_decoupling -> no clean decoupling story to compare.
VERDICT_TO_GROUP = {
    'genuine':      'Genuine',
    'fake':         'Fake',
    'net_exporter': 'Special',
}
classes['group'] = classes['verdict'].map(VERDICT_TO_GROUP)
classes = classes.dropna(subset=['group'])

# Drop data-quality caveats (IRL GDP distortion, LUX, MLT) so a distorted
# elasticity can't pollute a group. Flip to False to keep them.
EXCLUDE_CAVEATED = True
if EXCLUDE_CAVEATED:
    classes = classes[classes['caveat'].isna()]

VERDICT = dict(zip(classes['iso_code'], classes['group']))
FOCUS = list(classes['iso_code'])

GROUP_COLORS = {
    'Genuine': '#2ecc71',
    'Fake':    '#e74c3c',
    'Special': '#f39c12',
}

print(f"Loaded {len(FOCUS)} classified countries from decoupler_class")
print(classes['group'].value_counts().to_string())

2026-07-03 17:29:56,342 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-07-03 17:29:56,344 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2026-07-03 17:29:56,345 INFO sqlalchemy.engine.Engine [cached since 186.6s ago] {'table_name': <sqlalchemy.sql.elements.TextClause object at 0x0000015C1EC5FC50>, 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2026-07-03 17:29:56,347 INFO sqlalchemy.engine.Engine 
    SELECT 
        iso_code, 
        country, 
        verdict, 
        caveat 
    FROM 
        decoupler_class

2

In [36]:
# Load all city stats for focus countries
query = """
    SELECT
        ci.city_code,
        ci.name AS city_name,
        ci.population,
        co.name AS country,
        co.iso_code,
        co.income_group,
        cs.year,
        cs.cars_per_1000,
        cs.bicycle_network_km,
        cs.municipal_waste_1000t,
        cs.avg_journey_to_work_min,
        cs.public_transport_cost_eur,
        cs.industrial_land_pct
    FROM 
        city_stats cs
    JOIN 
        cities ci ON ci.id = cs.city_id
    JOIN 
        countries co ON co.id = ci.country_id
    WHERE 
        co.iso_code = ANY(:countries)
    ORDER BY 
        co.iso_code, 
        ci.city_code, 
        cs.year
"""

with engine.connect() as conn:
    df = pd.read_sql(text(query), conn, params={'countries': FOCUS})

df['group'] = df['iso_code'].map(VERDICT)

print(f"Rows: {len(df)}")
print(f"Cities: {df['city_code'].nunique()}")
print(f"Countries: {df['iso_code'].nunique()}")
print(f"Years: {df['year'].min()} - {df['year'].max()}")
print()
print("Cities per country:")
print(
    df.groupby(['iso_code', 'group'])['city_code']
    .nunique()
    .reset_index()
    .rename(columns={'city_code': 'n_cities'})
    .sort_values('group')
    .to_string(index=False)
)

2026-07-03 17:29:56,386 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-07-03 17:29:56,387 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2026-07-03 17:29:56,387 INFO sqlalchemy.engine.Engine [cached since 186.6s ago] {'table_name': <sqlalchemy.sql.elements.TextClause object at 0x0000015C2421FA10>, 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2026-07-03 17:29:56,387 INFO sqlalchemy.engine.Engine 
    SELECT
        ci.city_code,
        ci.name AS city_name,
        ci.population,
        co.name AS country,
     

In [37]:
# Coverage check
INDICATORS = [
    'cars_per_1000', 
    'bicycle_network_km',
    'municipal_waste_1000t'
]

print("Non-null rows per indicator per group:")
for ind in INDICATORS:
    row = f"  {ind:<30}"
    for group in ['Genuine', 'Fake', 'Special']:
        n = df[df['group'] == group][ind].notna().sum()
        row += f"  {group}: {n:>5}"
    print(row)

years_coverage = (
    df[df['cars_per_1000'].notna()]
    .groupby('year')['city_code'].nunique()
    .sort_values(ascending=False)
    .head(8)
)

print("\nBest coverage years for cars_per_1000:")
print(years_coverage.to_string())

Non-null rows per indicator per group:
  cars_per_1000                   Genuine:  5835  Fake:  1480  Special:   613
  bicycle_network_km              Genuine:  3305  Fake:   252  Special:   609
  municipal_waste_1000t           Genuine:  4806  Fake:  1406  Special:   815

Best coverage years for cars_per_1000:
year
2018    564
2017    561
2011    520
2016    495
2015    494
2021    473
2020    469
2019    469


In [38]:
# Pick best snapshot year per indicator (highest city coverage)
def best_year(df, col):
    return (
        df[df[col].notna()]
        .groupby('year')['city_code'].nunique()
        .idxmax()
    )

YEAR_CARS  = best_year(df, 'cars_per_1000')
YEAR_WASTE = best_year(df, 'municipal_waste_1000t')
YEAR_BIKE  = best_year(df, 'bicycle_network_km')

print(f"Best snapshot year - cars:  {YEAR_CARS}")
print(f"Best snapshot year - waste: {YEAR_WASTE}")
print(f"Best snapshot year - bike:  {YEAR_BIKE}")

snap_cars  = df[(df['year'] == YEAR_CARS)  & df['cars_per_1000'].notna()].copy()
snap_waste = df[(df['year'] == YEAR_WASTE) & df['municipal_waste_1000t'].notna()].copy()
snap_bike  = df[(df['year'] == YEAR_BIKE)  & df['bicycle_network_km'].notna()].copy()

Best snapshot year - cars:  2018
Best snapshot year - waste: 2011
Best snapshot year - bike:  2020


In [39]:
# CARS PER 1000: the main story
# Hypothesis: cities in genuine decouplers are less car-dependent

# Box plot: distribution by group
fig = px.box(
    snap_cars,
    x='group',
    y='cars_per_1000',
    color='group',
    points='all',
    hover_name='city_name',
    hover_data=['country', 'iso_code'],
    color_discrete_map=GROUP_COLORS,
    category_orders={'group': ['Genuine', 'Fake', 'Special']},
    title=f'Cars per 1000 population - Genuine vs Fake Decouplers ({YEAR_CARS})<br>'
          f'<sup>Do cities in genuine decouplers own fewer cars?</sup>',
    labels={'cars_per_1000': 'Cars per 1000 population', 'group': 'Decoupling group'},
    height=550
)
fig.show()

# Summary stats
print(f"Cars per 1000 by group ({YEAR_CARS}):")
print(
    snap_cars.groupby('group')['cars_per_1000']
    .agg(['median', 'mean', 'std', 'count'])
    .round(1)
    .to_string()
)

Cars per 1000 by group (2018):
         median   mean    std  count
group                               
Fake      428.7  426.0   72.5    167
Genuine   467.7  493.3  198.2    344
Special   557.2  568.7   89.3     53


In [40]:
# Median cars per country - ranked
country_cars = (
    snap_cars.groupby(['iso_code', 'country', 'group'])['cars_per_1000']
    .median()
    .reset_index()
    .sort_values('cars_per_1000')
)

fig = px.bar(
    country_cars,
    x='iso_code',
    y='cars_per_1000',
    color='group',
    color_discrete_map=GROUP_COLORS,
    category_orders={
        'iso_code': country_cars['iso_code'].tolist(),
        'group': ['Genuine', 'Fake', 'Special']
    },
    title=f'Median Cars per 1000 by Country ({YEAR_CARS})<br>'
          '<sup>Ranked lowest to highest - colored by decoupling verdict</sup>',
    labels={'cars_per_1000': 'Median cars per 1000', 'iso_code': 'Country'},
    height=500
)
fig.show()

print(country_cars[['iso_code', 'group', 'cars_per_1000']].to_string(index=False))

iso_code   group  cars_per_1000
     LVA    Fake        277.110
     EST Genuine        333.600
     SVK Genuine        343.115
     SWE Genuine        355.240
     FIN Genuine        379.890
     HUN Genuine        382.550
     HRV    Fake        395.320
     LTU Genuine        406.570
     CHE    Fake        419.980
     BEL    Fake        422.970
     DEU Genuine        430.960
     GBR    Fake        434.880
     SVN Genuine        438.830
     FRA Genuine        495.710
     POL Special        557.230
     ITA Genuine        618.350


In [41]:
# Cars per 1000 - trend over time by group
# Are genuine decouplers reducing car dependency faster?

trend = (
    df[df['cars_per_1000'].notna()]
    .groupby(['year', 'group'])['cars_per_1000']
    .median()
    .reset_index()
)

fig = px.line(
    trend,
    x='year',
    y='cars_per_1000',
    color='group',
    color_discrete_map=GROUP_COLORS,
    markers=True,
    title='Cars per 1000 - Median Trend by Decoupling Group<br>'
          '<sup>Are genuine decouplers reducing car dependency faster?</sup>',
    labels={'cars_per_1000': 'Median cars per 1000', 'year': 'Year'},
    height=500
)
fig.show()

# Trend per country
country_trend = (
    df[df['cars_per_1000'].notna()]
    .groupby(['year', 'iso_code', 'group'])['cars_per_1000']
    .median()
    .reset_index()
)

fig2 = px.line(
    country_trend,
    x='year',
    y='cars_per_1000',
    color='iso_code',
    line_dash='group',
    title='Cars per 1000 - per Country over Time<br>'
          '<sup>Dotted = fake decoupler | Solid = genuine | Dashed = special</sup>',
    labels={'cars_per_1000': 'Median cars per 1000', 'year': 'Year'},
    height=550
)
fig2.show()

In [42]:
# Bicycle network

country_bike = (
    snap_bike.groupby(['iso_code', 'country', 'group'])['bicycle_network_km']
    .median()
    .reset_index()
    .sort_values('bicycle_network_km', ascending=False)
)

fig = px.bar(
    country_bike,
    x='iso_code',
    y='bicycle_network_km',
    color='group',
    color_discrete_map=GROUP_COLORS,
    category_orders={
        'iso_code': country_bike['iso_code'].tolist(),
        'group': ['Genuine', 'Fake', 'Special']
    },
    title=f'Median Bicycle Network km per City ({YEAR_BIKE})<br>'
          '<sup>Countries that did not report this indicator are absent</sup>',
    labels={'bicycle_network_km': 'Median bicycle network (km)', 'iso_code': 'Country'},
    height=500
)
fig.show()

print(f"Countries with bicycle data in {YEAR_BIKE}: {sorted(snap_bike['iso_code'].unique())}")
print(f"Countries WITHOUT bicycle data: {sorted(set(FOCUS) - set(snap_bike['iso_code'].unique()))}")
print()
print(country_bike[['iso_code', 'group', 'bicycle_network_km']].to_string(index=False))

Countries with bicycle data in 2020: ['BEL', 'BGR', 'DEU', 'EST', 'FIN', 'HRV', 'HUN', 'ITA', 'LTU', 'LVA', 'NLD', 'POL', 'SVK', 'SVN']
Countries WITHOUT bicycle data: ['ALB', 'BLR', 'CHE', 'CZE', 'DNK', 'FRA', 'GBR', 'GEO', 'GRC', 'PRT', 'ROU', 'SWE']

iso_code   group  bicycle_network_km
     NLD Genuine             852.000
     FIN Genuine             594.000
     SVN Genuine             213.000
     BEL    Fake             207.700
     EST Genuine             129.000
     DEU Genuine             121.000
     LTU Genuine              93.650
     POL Special              49.350
     LVA    Fake              44.815
     ITA Genuine              29.100
     SVK Genuine              27.585
     HUN Genuine              23.600
     HRV    Fake              20.000
     BGR Genuine               7.075


In [43]:
# Municipal waste
# More waste -> higher consumption -> less efficient city

# Normalize by population to get waste per capita
snap_waste['waste_per_capita'] = (
    snap_waste['municipal_waste_1000t'] * 1000 /  # convert to tonnes
    snap_waste['population'].replace(0, np.nan)
)

# Use raw totals grouped by country (per capita is noisy if population missing)
fig = px.box(
    snap_waste[snap_waste['municipal_waste_1000t'].notna()],
    x='group',
    y='municipal_waste_1000t',
    color='group',
    points='all',
    hover_name='city_name',
    hover_data=['country'],
    color_discrete_map=GROUP_COLORS,
    category_orders={'group': ['Genuine', 'Fake', 'Special']},
    title=f'Municipal Waste per City ({YEAR_WASTE}, 1000 tonnes)<br>'
          '<sup>Higher waste = more consumption - do fake decouplers waste more?</sup>',
    labels={'municipal_waste_1000t': 'Municipal waste (1000 tonnes)', 'group': 'Group'},
    height=550,
    log_y=True
)
fig.show()

print(f"\nMunicipal waste by group ({YEAR_WASTE}, 1000 tonnes):")
print(
    snap_waste.groupby('group')['municipal_waste_1000t']
    .agg(['median', 'mean', 'count'])
    .round(1)
    .to_string()
)


Municipal waste by group (2011, 1000 tonnes):
         median   mean  count
group                        
Fake       77.8   96.1    140
Genuine    62.0  113.9    382
Special    38.8   61.3     68


In [44]:
# Cars vs waste
# Cities with low cars AND low waste are structurally efficient

# Use the year with best overlap between the two indicators
YEAR_BOTH = (
    df[df['cars_per_1000'].notna() & df['municipal_waste_1000t'].notna()]
    .groupby('year')['city_code'].nunique()
    .idxmax()
)
print(f"Best year for cars + waste overlap: {YEAR_BOTH}")

snap_both = df[
    (df['year'] == YEAR_BOTH) &
    df['cars_per_1000'].notna() &
    df['municipal_waste_1000t'].notna()
].copy()

print(f"Cities with both indicators: {len(snap_both)}")

fig = px.scatter(
    snap_both,
    x='cars_per_1000',
    y='municipal_waste_1000t',
    color='group',
    color_discrete_map=GROUP_COLORS,
    hover_name='city_name',
    hover_data=['country', 'iso_code'],
    size='population',
    size_max=40,
    log_y=True,
    title=f'Cars per 1000 vs Municipal Waste - City structural fingerprint ({YEAR_BOTH})<br>',
    labels={
        'cars_per_1000': 'Cars per 1000 population',
        'municipal_waste_1000t': 'Municipal waste (1000 tonnes, log scale)',
    },
    category_orders={'group': ['Genuine', 'Fake', 'Special']},
    height=600
)
fig.show()

Best year for cars + waste overlap: 2017
Cities with both indicators: 497


In [45]:
# High-income genuine vs high-income fake
hi_df = snap_cars[
    (snap_cars['income_group'] == 'High income') &
    (snap_cars['group'].isin(['Genuine', 'Fake']))
].copy()
hi_df['hi_group'] = hi_df['group'].map({
    'Genuine': 'Genuine (high income)',
    'Fake':    'Fake (high income)',
})

hi_colors = {
    'Genuine (high income)': '#2ecc71',
    'Fake (high income)':    '#e74c3c',
}

print("High-income countries in each group:")
print(hi_df.groupby('hi_group')['iso_code'].unique().to_string())

fig = px.box(
    hi_df,
    x='iso_code',
    y='cars_per_1000',
    color='hi_group',
    points='all',
    hover_name='city_name',
    color_discrete_map=hi_colors,
    title=f'Cars per 1000 - High-Income Genuine vs Fake Decouplers ({YEAR_CARS})<br>'
          '<sup>Do genuine decouplers build differently?</sup>',
    labels={'cars_per_1000': 'Cars per 1000', 'iso_code': 'Country'},
    height=500
)
fig.show()

print(f"\nHigh-income group comparison ({YEAR_CARS}):")
print(
    hi_df.groupby('hi_group')['cars_per_1000']
    .agg(['median', 'mean', 'count'])
    .round(1)
    .to_string()
)

High-income countries in each group:
hi_group
Fake (high income)                               [BEL, CHE, GBR, HRV, LVA]
Genuine (high income)    [DEU, EST, FIN, FRA, HUN, ITA, LTU, SVK, SVN, ...



High-income group comparison (2018):
                       median   mean  count
hi_group                                   
Fake (high income)      428.7  426.0    167
Genuine (high income)   467.7  493.3    344


In [46]:
pol = df[df['iso_code'] == 'POL'].copy()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f'Cars per 1000 - Poland cities over time',
        f'Bicycle network km - Poland cities ({YEAR_BIKE})'
    ]
)

# Cars trend for all Polish cities
pol_cars = pol[pol['cars_per_1000'].notna()]
for city in pol_cars['city_code'].unique():
    city_data = pol_cars[pol_cars['city_code'] == city]
    fig.add_trace(
        go.Scatter(
            x=city_data['year'], 
            y=city_data['cars_per_1000'],
            mode='lines', 
            opacity=0.4,
            name=city_data['city_name'].iloc[0] or city,
            showlegend=False, 
            line=dict(color='#f39c12')
        ), row=1, col=1
    )

# Poland median
pol_median = pol_cars.groupby('year')['cars_per_1000'].median().reset_index()
fig.add_trace(
    go.Scatter(
        x=pol_median['year'], 
        y=pol_median['cars_per_1000'],
        mode='lines', 
        name='Poland median',
        line=dict(color='#e67e22', width=3)
    ), row=1, col=1
)

# Bike network
pol_bike = pol[(pol['year'] == YEAR_BIKE) & pol['bicycle_network_km'].notna()]
pol_bike_sorted = pol_bike.sort_values('bicycle_network_km', ascending=True)
fig.add_trace(
    go.Bar(
        x=pol_bike_sorted['bicycle_network_km'],
        y=pol_bike_sorted['city_name'].fillna(pol_bike_sorted['city_code']),
        orientation='h',
        marker_color='#f39c12',
        showlegend=False
    ), row=1, col=2
)

fig.update_layout(
    height=500,
    title_text='Poland city profile'
)
fig.show()

print(f"Poland - {pol_bike['city_code'].nunique()} cities with bicycle data in {YEAR_BIKE}")
print(f"Median bicycle network: {pol_bike['bicycle_network_km'].median():.0f} km")

Poland - 68 cities with bicycle data in 2020
Median bicycle network: 49 km


## What actually drives car dependency? City size (not the decoupling verdict)

The group comparison above found no signal - so what does explain why some European cities are more car-dependent? The urban-economics literature points to **city size / density and income**, not a country's emissions-accounting honesty:

- **Newman & Kenworthy** - a strong power-law: per-capita car use / transport energy *falls* as city size and density rise (bigger, denser cities support transit and shorter trips).
- **Creutzig et al. 2015 (PNAS)** - across 274 cities, transport energy is driven by **economic activity, fuel price and population density** (around 88% of urban transport energy), splitting cities into 8 types.
- **Eurostat** itself notes car ownership is *"linked to income level and population density"*, ranging 261 (Romania) → 625 (Italy) per 1000 - matching my per-country ranking almost exactly.
- Caveat from the literature: in **Europe CO2 scales roughly *linearly* with population** (unlike the superlinear US), so the size effect is real but *weaker* here.

**The honest test:** does city **size** explain car dependency, and does the **decoupling verdict add anything beyond size and national context**? A caveat on income: every city in my Eurostat car-data sample sits in a **high-income** country (Romania and Bulgaria are now World-Bank "high income" as well), so income has no in-sample variation to regress on - its effect is visible only at the **country** level. So I test size directly, remove the national confound (within-country), and finish with a nested regression against city size and country identity.

In [47]:
# Size bands: does car dependency fall with city size? (Newman-Kenworthy)
SIZE_BINS   = [0, 100_000, 250_000, 500_000, 1_000_000, np.inf]
SIZE_LABELS = ['<100k', '100-250k', '250-500k', '500k-1M', '>1M']

sc = snap_cars.dropna(subset=['cars_per_1000', 'population']).copy()
sc['size_band'] = pd.cut(sc['population'], bins=SIZE_BINS, labels=SIZE_LABELS)

print(f"City population range: {sc['population'].min():,.0f} - {sc['population'].max():,.0f}")
print("Cities per size band:")
print(sc['size_band'].value_counts().reindex(SIZE_LABELS).to_string())

fig = px.box(
    sc, x='size_band', y='cars_per_1000',
    category_orders={'size_band': SIZE_LABELS},
    points='all', hover_name='city_name', hover_data=['country'],
    title=f'Cars per 1000 by city-size band ({YEAR_CARS})<br>'
          '<sup>Newman-Kenworthy: do bigger cities carry fewer cars per person?</sup>',
    labels={'cars_per_1000': 'Cars per 1000', 'size_band': 'City population'},
    height=500
)
fig.show()

print(f"\nCars per 1000 by size band ({YEAR_CARS}):")
print(sc.groupby('size_band', observed=True)['cars_per_1000']
        .agg(['median', 'mean', 'count']).round(1).to_string())

City population range: 37,960 - 10,353,710
Cities per size band:
size_band
<100k       175
100-250k    261
250-500k     77
500k-1M      36
>1M          15



Cars per 1000 by size band (2018):
           median   mean  count
size_band                      
<100k       493.0  501.9    175
100-250k    468.2  492.4    261
250-500k    420.4  432.6     77
500k-1M     383.8  420.0     36
>1M         357.9  413.7     15


In [48]:
# Control for the national confound: does size matter within a country?
# Demean cars and log(pop) by country, then correlate the residuals.
from scipy.stats import pearsonr

wc = sc.copy()
wc['log_pop']    = np.log(wc['population'])
wc['cars_dm']    = wc['cars_per_1000'] - wc.groupby('iso_code')['cars_per_1000'].transform('mean')
wc['log_pop_dm'] = wc['log_pop']       - wc.groupby('iso_code')['log_pop'].transform('mean')

pooled_r, pooled_p = pearsonr(wc['log_pop'], wc['cars_per_1000'])
within_r, within_p = pearsonr(wc['log_pop_dm'], wc['cars_dm'])

print("Cars per 1000 vs log(population):")
print(f"  Pooled (all cities):                        r = {pooled_r:+.2f}  (p = {pooled_p:.3f})")
print(f"  Within-country (national confound removed): r = {within_r:+.2f}  (p = {within_p:.3f})")

# Per-country size slope, for countries with enough cities.
rows = []
for iso, g in wc.groupby('iso_code'):
    if g['city_code'].nunique() >= 8:
        r, p = pearsonr(g['log_pop'], g['cars_per_1000'])
        rows.append({'iso_code': iso, 'n_cities': g['city_code'].nunique(),
                     'r_cars_vs_logpop': round(r, 2), 'p': round(p, 3)})
print("\nPer-country size effect (countries with >=8 cities):")
print(pd.DataFrame(rows).sort_values('r_cars_vs_logpop').to_string(index=False))

Cars per 1000 vs log(population):
  Pooled (all cities):                        r = -0.18  (p = 0.000)
  Within-country (national confound removed): r = -0.15  (p = 0.000)

Per-country size effect (countries with >=8 cities):
iso_code  n_cities  r_cars_vs_logpop     p
     FIN         9             -0.96 0.000
     SWE        13             -0.78 0.002
     DEU       127             -0.54 0.000
     GBR       135             -0.51 0.000
     BEL        11             -0.47 0.144
     CHE        10             -0.40 0.246
     FRA        76             -0.25 0.030
     HUN        19             -0.17 0.497
     ITA        87             -0.08 0.479
     POL        53              0.35 0.010


In [49]:
# Newman-Kenworthy, visualized: cars vs city size, with an overall OLS trend.
fig = px.scatter(
    sc, x='population', y='cars_per_1000', color='group',
    color_discrete_map=GROUP_COLORS, hover_name='city_name', hover_data=['country'],
    log_x=True, trendline='ols', trendline_scope='overall', trendline_color_override='black',
    category_orders={'group': ['Genuine', 'Fake', 'Special']},
    title=f'Cars per 1000 vs city population ({YEAR_CARS}, log x)<br>'
          '<sup>Bigger cities carry fewer cars per person</sup>',
    labels={'population': 'City population (log scale)', 'cars_per_1000': 'Cars per 1000'},
    height=550
)
fig.show()

# Income can't be tested in-sample: every car-data city is in a high-income country.
print("Income groups among cities with car data:")
print(sc['income_group'].value_counts(dropna=False).to_string())
print("\n-> Income is ~constant here, so within our Eurostat sample the measurable drivers are city size"
      "\n   and national context. The income effect is a country-level observation (Eurostat: RO 261 -> IT 625).")

Income groups among cities with car data:
income_group
High income    564

-> Income is ~constant here, so within our Eurostat sample the measurable drivers are city size
   and national context. The income effect is a country-level observation (Eurostat: RO 261 -> IT 625).


In [50]:
# Is the decoupling verdict a real urban-form driver, or just a country label?
# Nested OLS. NOTE: every city with car data here is in a HIGH-INCOME country, so income
# has no in-sample variation - the testable drivers are city size and country identity.
import statsmodels.formula.api as smf

reg_df = sc.dropna(subset=['cars_per_1000', 'population', 'group']).copy()
reg_df['log_pop'] = np.log(reg_df['population'])

m_size    = smf.ols('cars_per_1000 ~ log_pop', data=reg_df).fit()
m_group   = smf.ols('cars_per_1000 ~ log_pop + C(group)', data=reg_df).fit()
m_country = smf.ols('cars_per_1000 ~ log_pop + C(iso_code)', data=reg_df).fit()

print(f"income groups among car-data cities: {reg_df['income_group'].dropna().unique().tolist()}  (no variation to test)\n")
print(f"cars ~ log(pop)                     : R2 = {m_size.rsquared:.3f}   (n={int(m_size.nobs)})")
print(f"cars ~ log(pop) + decoupling verdict: R2 = {m_group.rsquared:.3f}   (dR2 = {m_group.rsquared - m_size.rsquared:+.3f})")
print(f"cars ~ log(pop) + COUNTRY           : R2 = {m_country.rsquared:.3f}   (dR2 = {m_country.rsquared - m_size.rsquared:+.3f})")
print(f"\nSize effect: {m_size.params['log_pop']:+.0f} cars/1000 per e-fold of population "
      f"(p = {m_size.pvalues['log_pop']:.3f}) - bigger cities, fewer cars.")
print("\nVerdict coefficients (baseline = Fake):")
for t in m_group.params.index:
    if t.startswith('C(group)'):
        print(f"  {t:<22} {m_group.params[t]:+7.1f}  (p = {m_group.pvalues[t]:.3f})")
print("""
Read: the verdict is statistically 'significant' - but note the sign is wrong for the original
hypothesis (genuine cities have MORE cars, not fewer), and it explains far less than country identity.
Because the verdict is a country-level label (constant within each country) it is nested inside COUNTRY
and carries no city-structure information beyond it. Its apparent effect is just national composition
(the genuine group happens to contain high-car Italy & France) - the Part-A confound in regression
form, not an urban-form mechanism.""")

income groups among car-data cities: ['High income']  (no variation to test)

cars ~ log(pop)                     : R2 = 0.032   (n=564)
cars ~ log(pop) + decoupling verdict: R2 = 0.085   (dR2 = +0.053)
cars ~ log(pop) + COUNTRY           : R2 = 0.334   (dR2 = +0.302)

Size effect: -37 cars/1000 per e-fold of population (p = 0.000) - bigger cities, fewer cars.

Verdict coefficients (baseline = Fake):
  C(group)[T.Genuine]      +59.7  (p = 0.000)
  C(group)[T.Special]     +135.6  (p = 0.000)

Read: the verdict is statistically 'significant' - but note the sign is wrong for the original
hypothesis (genuine cities have MORE cars, not fewer), and it explains far less than country identity.
Because the verdict is a country-level label (constant within each country) it is nested inside COUNTRY
and carries no city-structure information beyond it. Its apparent effect is just national composition
(the genuine group happens to contain high-car Italy & France) - the Part-A confound in regression


In [52]:
print("=" * 30)
print("NOTEBOOK 07 - KEY FINDINGS")
print("=" * 30)

# --- Part A: the decoupling split (numbers from the group snapshots above) ---
med_cars = snap_cars.groupby('group')['cars_per_1000'].median()
hi_med   = hi_df.groupby('hi_group')['cars_per_1000'].median()
med_wpc  = snap_waste.groupby('group')['waste_per_capita'].median()
silent         = sorted(set(FOCUS) - set(snap_bike['iso_code'].unique()))
silent_genuine = [i for i in silent if VERDICT.get(i) == 'Genuine']

# --- Part B: the real drivers (numbers from the size section) ---
size_med = sc.groupby('size_band', observed=True)['cars_per_1000'].median()

print(f"""
PART A - Does the genuine/fake decoupling split explain city structure?  NO.
   Cars per 1000 ({YEAR_CARS}):        Genuine {med_cars.get('Genuine', float('nan')):.0f}  |  Fake {med_cars.get('Fake', float('nan')):.0f}   (genuine slightly HIGHER, not lower)
   Same, high-income only:             Genuine {hi_med.get('Genuine (high income)', float('nan')):.0f}  |  Fake {hi_med.get('Fake (high income)', float('nan')):.0f}
   Municipal waste per capita ({YEAR_WASTE}): Genuine {med_wpc.get('Genuine', float('nan')):.2f}  |  Fake {med_wpc.get('Fake', float('nan')):.2f}  t/person   (no clean gap)
   Bike-network 'silence' splits both groups (genuine & silent: {silent_genuine}) - a Eurostat
   coverage artifact.

PART B - So what does drive car dependency?  City size (and national context).
   Cars per 1000 by size band ({YEAR_CARS}):""")
for b in SIZE_LABELS:
    if b in size_med.index and pd.notna(size_med[b]):
        print(f"      {b:<9} {size_med[b]:.0f}")
print(f"""   Size effect (cars vs log-population):  pooled r = {pooled_r:+.2f},  within-country r = {within_r:+.2f}  (p<0.001)
      -> Newman-Kenworthy holds (weakly, as expected for Europe): bigger cities carry fewer cars per
         person, and it survives removing the national confound. This is the one real urban-form signal.
   Income is not testable here - every car-data city is in a high-income country (no in-sample variation).
      The income effect is a country-level observation (Eurostat: RO 261 -> IT 625).
   Regression:  cars ~ log(pop) R2 = {m_size.rsquared:.2f}  |  + verdict R2 = {m_group.rsquared:.2f}  |  + country R2 = {m_country.rsquared:.2f}
      -> The verdict is statistically significant but with the WRONG sign (genuine = more cars) and is
         nested within country - it proxies national car culture, not urban form. Country identity
         explains far more; the verdict adds nothing city-structural on top.

SUMMARY
   The only genuine urban-form signal is city size (bigger -> fewer cars per person; Newman-Kenworthy,
   Creutzig et al.). Decoupling honesty does not shape city structure: the verdict tracks national car
   culture and even points opposite to the original 'greener cities' hypothesis. The offshoring signal
   is real - but it lives at the country level (consumption vs territorial CO2, NB04), not in cities.
   Poland stays as a descriptive spotlight (net exporter, extensive bike data), as an individual case.
""")

NOTEBOOK 07 - KEY FINDINGS

PART A - Does the genuine/fake decoupling split explain city structure?  NO.
   Cars per 1000 (2018):        Genuine 468  |  Fake 429   (genuine slightly HIGHER, not lower)
   Same, high-income only:             Genuine 468  |  Fake 429
   Municipal waste per capita (2011): Genuine 0.48  |  Fake 0.41  t/person   (no clean gap)
   Bike-network 'silence' splits both groups (genuine & silent: ['CZE', 'FRA', 'GRC', 'PRT', 'ROU', 'SWE']) - a Eurostat
   coverage artifact.

PART B - So what does drive car dependency?  City size (and national context).
   Cars per 1000 by size band (2018):
      <100k     493
      100-250k  468
      250-500k  420
      500k-1M   384
      >1M       358
   Size effect (cars vs log-population):  pooled r = -0.18,  within-country r = -0.15  (p<0.001)
      -> Newman-Kenworthy holds (weakly, as expected for Europe): bigger cities carry fewer cars per
         person, and it survives removing the national confound. This is the one rea